<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/07-pandas/02_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 7: Exercises

Twelve exercises: four Easy, five Medium and three Hard. They're the everyday jobs of working with tables: loading, summarising, filtering, filling, encoding, joining and reshaping.

Each check builds a small DataFrame, runs your function, and compares the result with the expected table: same columns, same index, same values (allowing for tiny floating-point differences). Small type differences between pandas versions, like `object` vs `str` for text, are ignored.

**Run the setup cell first.**

Rules:

- pandas, NumPy and the standard library only.
- No loops over rows (`for`, `iterrows`, `apply(axis=1)`). Use vectorized operations, `groupby`, `merge` and friends.
- If a check fails, it prints both tables. Compare the columns and the index first, then the values.
- Solutions are in `03_solutions.ipynb`.

In [ ]:
# Setup: run this cell once before you start.
# It defines the check functions used below.

import copy
import math
import inspect
import tempfile
import time
from pathlib import Path

import numpy as np
import pandas as pd
from io import StringIO


class Raises:
    """Marks a test case where the code is expected to raise an error."""
    def __init__(self, exc):
        self.exc = exc

    def __repr__(self):
        return f"{self.exc.__name__} to be raised"


def _same(a, b, tol=1e-6):
    if isinstance(a, (pd.DataFrame, pd.Series)) or isinstance(b, (pd.DataFrame, pd.Series)):
        if type(a) is not type(b):
            return False
        options = dict(check_dtype=False, check_exact=False, rtol=tol, atol=tol, check_index_type=False)
        try:
            if isinstance(a, pd.DataFrame):
                pd.testing.assert_frame_equal(a, b, check_column_type=False, **options)
            else:
                pd.testing.assert_series_equal(a, b, **options)
            return True
        except AssertionError:
            return False
    if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
        try:
            a_arr, b_arr = np.asarray(a), np.asarray(b)
        except Exception:
            return False
        if a_arr.shape != b_arr.shape:
            return False
        if a_arr.dtype.kind in "fc" or b_arr.dtype.kind in "fc":
            return bool(np.allclose(a_arr, b_arr, rtol=tol, atol=tol))
        return bool(np.array_equal(a_arr, b_arr))
    if isinstance(a, (float, np.floating)) or isinstance(b, (float, np.floating)):
        if not isinstance(a, (int, float, np.number)) or not isinstance(b, (int, float, np.number)):
            return False
        if math.isnan(a) or math.isnan(b):
            return False
        return math.isclose(a, b, rel_tol=tol, abs_tol=tol)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(_same(x, y) for x, y in zip(a, b))
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_same(a[k], b[k]) for k in a)
    return a == b


def _short(x, limit=90):
    if isinstance(x, (pd.DataFrame, pd.Series)):
        lines = (x.to_string(max_rows=6, max_cols=6) if isinstance(x, pd.DataFrame) else x.to_string(max_rows=6)).splitlines()
        return f"{type(x).__name__} shape={x.shape}\n          " + "\n          ".join(lines[:8])
    if isinstance(x, np.ndarray):
        body = np.array2string(x, precision=4, threshold=12, edgeitems=3, separator=", ")
        text = f"array(shape={x.shape}, dtype={x.dtype}, {' '.join(body.split())})"
    elif isinstance(x, tuple) and any(isinstance(i, np.ndarray) for i in x):
        text = "(" + ", ".join(_short(i, limit) for i in x) + ")"
        return text
    else:
        text = repr(x)
    if isinstance(x, (pd.DataFrame, pd.Series)):
        return text
    return text if len(text) <= limit else text[:limit - 3] + "..."


def _scenarios(name, cases):
    for description, scenario, expected, hint in cases:
        try:
            result = scenario()
        except Exception as e:
            if isinstance(expected, Raises) and isinstance(e, expected.exc):
                continue
            print(f"FAIL  {name}: {description}")
            print(f"      raised {type(e).__name__}: {e}")
            if isinstance(expected, Raises):
                print(f"      expected {expected!r}")
            else:
                print(f"      expected: {_short(expected)}")
            if hint:
                print(f"      hint: {hint}")
            return
        if isinstance(expected, Raises):
            print(f"FAIL  {name}: {description}")
            print(f"      expected {expected!r}, but nothing was raised")
            if hint:
                print(f"      hint: {hint}")
            return
        if not _same(result, expected):
            print(f"FAIL  {name}: {description}")
            print(f"      expected: {_short(expected)}")
            print(f"      got:      {_short(result)}")
            if hint:
                print(f"      hint: {hint}")
            return
    print(f"PASS  all {len(cases)} tests passed for {name}")


# ---- checks ----

def _penguins_small():
    return pd.DataFrame({
        "species": ["Adelie", "Adelie", "Gentoo", "Gentoo", "Chinstrap", "Adelie", "Gentoo"],
        "island": ["Dream", "Torgersen", "Biscoe", "Biscoe", "Dream", "Biscoe", "Biscoe"],
        "bill_length_mm": [39.0, 40.0, 47.0, 50.0, 49.0, 38.0, 46.0],
        "bill_depth_mm": [19.5, 20.0, 15.0, 16.0, 17.5, 19.0, 14.0],
        "flipper_length_mm": [190.0, 195.0, 215.0, 225.0, 196.0, 185.0, 210.0],
        "body_mass_g": [3700.0, 4000.0, 5000.0, 5600.0, 3600.0, 3500.0, np.nan],
        "sex": ["male", "female", "male", np.nan, "female", "male", "female"],
    })


def check_load_scores(f):
    text = "id,date,score\n007,2025-01-03,0.9\n012,2025-02-10,?\n101,2025-02-11,-\n230,2025-03-01,0.75\n"

    def load():
        path = Path(tempfile.mkdtemp()) / "scores.csv"
        path.write_text(text, encoding="utf-8")
        return f(path)

    _scenarios("load_scores", [
        ("ids keep their leading zeros", lambda: load()["id"].tolist(), ["007", "012", "101", "230"],
         "pass dtype={'id': str} to read_csv"),
        ("dates are real dates", lambda: pd.api.types.is_datetime64_any_dtype(load()["date"]), True, "parse_dates=['date']"),
        ("'?' and '-' are missing", lambda: load()["score"].isna().tolist(), [False, True, True, False],
         "na_values=['?', '-']"),
        ("score is a float column", lambda: load()["score"].dtype.kind, "f", None),
    ])


def check_missing_summary(f):
    df = pd.DataFrame({"a": [1, None, None, 4], "b": [None, 2, 3, 4], "c": [1, 2, 3, 4], "d": [None, None, 1, 2]})
    expected = pd.DataFrame(
        {"n_missing": [2, 2, 1, 0], "pct_missing": [0.5, 0.5, 0.25, 0.0]},
        index=["a", "d", "b", "c"],
    )
    _scenarios("missing_summary", [
        ("counts, fractions and order", lambda: f(df), expected,
         "sort by n_missing (high to low), then by column name; the index holds the column names"),
        ("nothing missing", lambda: f(pd.DataFrame({"x": [1, 2]}))["n_missing"].tolist(), [0], None),
    ])


def check_filter_and_sort(f):
    df = _penguins_small()

    def unchanged():
        d = _penguins_small()
        f(d, "Adelie", 3600)
        return d.equals(_penguins_small())

    expected = pd.DataFrame({
        "species": ["Adelie", "Adelie"],
        "island": ["Torgersen", "Dream"],
        "body_mass_g": [4000.0, 3700.0],
    })
    _scenarios("filter_and_sort", [
        ("filter, keep three columns, sort, reset the index", lambda: f(df, "Adelie", 3600), expected,
         "the result should have a fresh index 0, 1, 2, ... (reset_index(drop=True))"),
        ("nothing matches", lambda: len(f(df, "Gentoo", 9000)), 0, None),
        ("missing masses are excluded", lambda: f(df, "Gentoo", 0)["body_mass_g"].tolist(), [5600.0, 5000.0], None),
        ("input is not modified", unchanged, True, None),
    ])


def check_add_features(f):
    def unchanged():
        d = _penguins_small()
        f(d)
        return list(d.columns) == list(_penguins_small().columns)

    _scenarios("add_features", [
        ("bill_ratio", lambda: f(_penguins_small())["bill_ratio"].round(4).tolist()[:3], [2.0, 2.0, 3.1333], None),
        ("mass_kg", lambda: f(_penguins_small())["mass_kg"].tolist()[:3], [3.7, 4.0, 5.0], None),
        ("missing mass stays missing", lambda: bool(f(_penguins_small())["mass_kg"].isna().iloc[-1]), True, None),
        ("all original columns are kept", lambda: list(f(_penguins_small()).columns)[:7], list(_penguins_small().columns), None),
        ("input is not modified", unchanged, True, "use assign, or copy() first"),
    ])


def check_species_table(f):
    expected = pd.DataFrame(
        {
            "n": [3, 3, 1],
            "mean_mass": [5300.0, 3733.3, 3600.0],
            "mean_flipper": [216.7, 190.0, 196.0],
            "pct_male": [0.5, 2 / 3, 0.0],
        },
        index=pd.Index(["Gentoo", "Adelie", "Chinstrap"], name="species"),
    )
    _scenarios("species_table", [
        ("named aggregation, rounding and order", lambda: f(_penguins_small()), expected,
         "n counts rows; means skip NaN; pct_male is the share of 'male' among penguins whose sex is known; sort by mean_mass, high to low"),
    ])


def check_fill_by_group(f):
    train = pd.DataFrame({"g": ["a", "a", "a", "b", "b", "c"], "x": [1.0, 3.0, np.nan, 10.0, 20.0, np.nan]})
    test = pd.DataFrame({"g": ["a", "b", "c", "d", "a"], "x": [np.nan, np.nan, np.nan, np.nan, 100.0]})

    def unchanged():
        tr, te = train.copy(), test.copy()
        f(tr, te, "x", "g")
        return tr.equals(train) and te.equals(test)

    _scenarios("fill_by_group", [
        ("training data: group medians", lambda: f(train, test, "x", "g")[0]["x"].tolist(), [1.0, 3.0, 2.0, 10.0, 20.0, 6.5],
         "compute the medians from train only; group c has no values at all in train, so it gets the overall training median (6.5)"),
        ("test data uses training medians", lambda: f(train, test, "x", "g")[1]["x"].tolist(), [2.0, 15.0, 6.5, 6.5, 100.0],
         "compute the medians from train only; groups with no training values, or unseen groups like 'd', get the overall training median"),
        ("inputs are not modified", unchanged, True, None),
    ])


def check_zscore_within_group(f):
    df = pd.DataFrame({"g": ["a", "a", "b", "b", "b", "c"], "x": [1.0, 3.0, 10.0, 10.0, 40.0, 7.0]}, index=[10, 11, 12, 13, 14, 15])
    s = np.sqrt(200)
    expected = pd.Series([-1.0, 1.0, -10 / s, -10 / s, 20 / s, 0.0], index=[10, 11, 12, 13, 14, 15], name="x")
    _scenarios("zscore_within_group", [
        ("z-score within each group, same index as the input", lambda: f(df, "x", "g"), expected,
         "use groupby(...).transform('mean') and transform('std', ddof=0); a group with std 0 gives 0"),
    ])


def check_one_hot_aligned(f):
    train = pd.DataFrame({"island": ["Biscoe", "Dream", "Biscoe"], "mass": [1.0, 2.0, 3.0]})
    test = pd.DataFrame({"island": ["Torgersen", "Dream"], "mass": [4.0, 5.0]})
    expected_train = pd.DataFrame({"mass": [1.0, 2.0, 3.0], "island_Biscoe": [1, 0, 1], "island_Dream": [0, 1, 0]})
    expected_test = pd.DataFrame({"mass": [4.0, 5.0], "island_Biscoe": [0, 0], "island_Dream": [0, 1]})
    _scenarios("one_hot_aligned", [
        ("training encoding", lambda: f(train, test, ["island"])[0], expected_train, "pd.get_dummies(..., columns=columns, dtype=int)"),
        ("test has exactly the training columns", lambda: f(train, test, ["island"])[1], expected_test,
         "reindex(columns=train_encoded.columns, fill_value=0): unseen categories are dropped, missing ones are 0"),
    ])


def check_top_n_per_group(f):
    df = pd.DataFrame({
        "g": ["b", "a", "b", "a", "b", "a"],
        "score": [10, 9, 7, 9, 1, 3],
        "name": ["p", "q", "r", "s", "t", "u"],
    })
    expected = pd.DataFrame({"g": ["a", "a", "b", "b"], "score": [9, 9, 10, 7], "name": ["q", "s", "p", "r"]})
    _scenarios("top_n_per_group", [
        ("top 2 per group", lambda: f(df, "g", "score", 2), expected,
         "sort by group, then by score (high to low) with a stable sort so ties keep their order; groupby(...).head(n); reset_index(drop=True)"),
        ("n larger than a group", lambda: len(f(df, "g", "score", 10)), 6, None),
    ])


def check_enrich_orders(f):
    orders = pd.DataFrame({"order_id": [3, 1, 2, 4], "customer_id": ["c2", "c1", "c9", "c1"], "amount": [5.0, 7.0, 1.0, 2.0]})
    customers = pd.DataFrame({"customer_id": ["c1", "c2", "c3"], "country": ["Nepal", "India", "Japan"]})
    dup = pd.concat([customers, customers.iloc[[0]]], ignore_index=True)
    expected = pd.DataFrame({
        "order_id": [3, 1, 2, 4],
        "customer_id": ["c2", "c1", "c9", "c1"],
        "amount": [5.0, 7.0, 1.0, 2.0],
        "country": ["India", "Nepal", np.nan, "Nepal"],
        "customer_found": [True, True, False, True],
    })
    _scenarios("enrich_orders", [
        ("left join, original order, found flag", lambda: f(orders, customers), expected,
         "how='left' keeps every order; customer_found is True where the customer exists"),
        ("duplicate customers are rejected", lambda: f(orders, dup), Raises(ValueError),
         "validate='many_to_one' raises a MergeError, which is a kind of ValueError"),
    ])


def check_rfm_features(f):
    orders = pd.DataFrame({
        "customer_id": ["a", "a", "b", "a", "c", "b"],
        "order_date": pd.to_datetime(["2025-01-01", "2025-01-20", "2025-01-25", "2025-02-05", "2025-02-10", "2025-01-30"]),
        "amount": [10.0, 20.0, 5.0, 100.0, 50.0, 15.0],
    })
    cutoff = pd.Timestamp("2025-02-01")
    expected = pd.DataFrame(
        {"recency_days": [12, 2], "frequency": [2, 2], "monetary": [30.0, 20.0]},
        index=pd.Index(["a", "b"], name="customer_id"),
    )
    _scenarios("rfm_features", [
        ("only orders before the cutoff count", lambda: f(orders, cutoff), expected,
         "filter order_date < cutoff first; orders on or after the cutoff are the future and must not leak in"),
    ])


def check_monthly_pivot(f):
    orders = pd.DataFrame({
        "customer_id": ["b", "a", "a", "b", "a"],
        "order_date": pd.to_datetime(["2025-01-15", "2025-01-02", "2025-01-30", "2025-04-01", "2025-04-20"]),
        "amount": [5.0, 10.0, 2.5, 7.0, 1.0],
    })
    expected = pd.DataFrame(
        {"2025-01": [12.5, 5.0], "2025-02": [0.0, 0.0], "2025-03": [0.0, 0.0], "2025-04": [1.0, 7.0]},
        index=pd.Index(["a", "b"], name="customer_id"),
    )
    expected.columns.name = "month"
    _scenarios("monthly_pivot", [
        ("every month from first to last, filled with 0", lambda: f(orders), expected,
         "months with no orders at all (Feb, Mar) still need columns: reindex with every month in the range"),
    ])


print("Setup done. Checks are ready.")

## Easy

### Exercise 1: `load_scores`

Write `load_scores(path)` that reads a CSV like this:

```
id,date,score
007,2025-01-03,0.9
012,2025-02-10,?
101,2025-02-11,-
```

and returns a DataFrame where:

- `id` stays text, keeping its leading zeros (`"007"`, not `7`)
- `date` is a real datetime column
- `score` is a float column, with `?` and `-` treated as missing

Everything can be done with arguments to `pd.read_csv`.

In [ ]:
import pandas as pd


def load_scores(path) -> pd.DataFrame:
    # your code here
    pass

In [ ]:
check_load_scores(load_scores)

### Exercise 2: `missing_summary`

Write `missing_summary(df)` that returns a DataFrame with one row per column of `df`:

- the **index** holds the column names
- `n_missing`: how many values are missing
- `pct_missing`: the fraction missing, from 0 to 1

Sort by `n_missing` from high to low, and break ties by column name.

This is the Module 5 `missing_report` exercise, in pandas.

In [ ]:
import pandas as pd


def missing_summary(df: pd.DataFrame) -> pd.DataFrame:
    # your code here
    pass

In [ ]:
check_missing_summary(missing_summary)

### Exercise 3: `filter_and_sort`

Write `filter_and_sort(df, species, min_mass)` for a penguin DataFrame. Return a new DataFrame with:

- only the rows of that `species` with `body_mass_g` at least `min_mass`
- only the columns `species`, `island` and `body_mass_g`
- sorted by `body_mass_g`, heaviest first
- a fresh index `0, 1, 2, ...`

Don't modify `df`.

In [ ]:
import pandas as pd


def filter_and_sort(df: pd.DataFrame, species: str, min_mass: float) -> pd.DataFrame:
    # your code here
    pass

In [ ]:
check_filter_and_sort(filter_and_sort)

### Exercise 4: `add_features`

Write `add_features(df)` that returns a **new** DataFrame with every original column plus two more:

- `bill_ratio`: bill length divided by bill depth
- `mass_kg`: body mass in kilograms

Missing values should stay missing. Don't modify `df`, and no loops.

In [ ]:
import pandas as pd


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    # your code here
    pass

In [ ]:
check_add_features(add_features)

## Medium

### Exercise 5: `species_table`

Write `species_table(df)` that summarises a penguin DataFrame with one row per species (the index, named `species`) and these columns:

- `n`: number of rows
- `mean_mass`: mean body mass, rounded to 1 decimal
- `mean_flipper`: mean flipper length, rounded to 1 decimal
- `pct_male`: the fraction of penguins that are `"male"`, among those whose sex is known (not rounded)

Sort by `mean_mass`, heaviest first.

In [ ]:
import pandas as pd


def species_table(df: pd.DataFrame) -> pd.DataFrame:
    # your code here
    pass

In [ ]:
check_species_table(species_table)

### Exercise 6: `fill_by_group`

Write `fill_by_group(train, test, column, group)` that fills missing values in `column` with the **median of that row's group**, computed from the **training data only**.

- If a group has no values in the training data, or appears only in the test data, use the overall training median of the column.
- Return `(train_filled, test_filled)`. Don't modify the inputs.

For example, a penguin with no recorded body mass gets the median mass of its own species, which is a much better guess than the median of all penguins.

In [ ]:
import pandas as pd


def fill_by_group(train: pd.DataFrame, test: pd.DataFrame, column: str, group: str):
    # your code here
    pass

In [ ]:
check_fill_by_group(fill_by_group)

### Exercise 7: `zscore_within_group`

Write `zscore_within_group(df, column, group)` that returns a Series, with the same index as `df`, giving each value's z-score **within its own group**:

$$z = \frac{x - \text{group mean}}{\text{group std}}$$

Use the population standard deviation (`ddof=0`). If a group's standard deviation is 0 (including groups with a single row), return 0 for those rows.

For example, "how heavy is this penguin *for its species*?" is often a more useful feature than its raw mass.

In [ ]:
import numpy as np
import pandas as pd


def zscore_within_group(df: pd.DataFrame, column: str, group: str) -> pd.Series:
    # your code here
    pass

In [ ]:
check_zscore_within_group(zscore_within_group)

### Exercise 8: `one_hot_aligned`

Write `one_hot_aligned(train, test, columns)` that one-hot encodes the given categorical columns, with the **training data defining the columns**:

- Encode `train` with `pd.get_dummies`, using integer 0/1 columns. Other columns are kept as they are.
- Give the encoded `test` **exactly** the same columns, in the same order: categories missing from the test data get all-zero columns, and categories that only appear in the test data are dropped.

Return `(train_encoded, test_encoded)`.

In [ ]:
import pandas as pd


def one_hot_aligned(train: pd.DataFrame, test: pd.DataFrame, columns: list[str]):
    # your code here
    pass

In [ ]:
check_one_hot_aligned(one_hot_aligned)

### Exercise 9: `top_n_per_group`

Write `top_n_per_group(df, group, by, n)` that returns the `n` rows with the highest values of `by` within each group.

- Order the result by group (A to Z), and within each group by `by` (highest first).
- If two rows tie on `by`, keep them in their original order.
- Give the result a fresh index `0, 1, 2, ...`.

For example: the three heaviest penguins of each species, or the top five products per category.

In [ ]:
import pandas as pd


def top_n_per_group(df: pd.DataFrame, group: str, by: str, n: int) -> pd.DataFrame:
    # your code here
    pass

In [ ]:
check_top_n_per_group(top_n_per_group)

## Hard

### Exercise 10: `enrich_orders`

Write `enrich_orders(orders, customers)` that adds customer information to a table of orders:

- Every order must be kept, in its original order, even if its customer isn't in `customers`.
- Add the customer columns, plus a boolean column `customer_found` that's `True` if the order's customer was found.
- If `customers` has the same `customer_id` more than once, raise a `ValueError` instead of silently duplicating orders.
- The result must have exactly as many rows as `orders`.

In [ ]:
import pandas as pd


def enrich_orders(orders: pd.DataFrame, customers: pd.DataFrame) -> pd.DataFrame:
    # your code here
    pass

In [ ]:
check_enrich_orders(enrich_orders)

### Exercise 11: `rfm_features`

**RFM** (recency, frequency, monetary) features are a classic way to describe customers from their purchase history. Write `rfm_features(orders, cutoff)`, where `orders` has columns `customer_id`, `order_date` (datetime) and `amount`, and `cutoff` is a `pd.Timestamp`.

Using **only orders strictly before `cutoff`**, return a DataFrame indexed by `customer_id` (sorted) with:

- `recency_days`: days from the customer's last order to the cutoff (an integer)
- `frequency`: number of orders
- `monetary`: total amount spent

Customers with no orders before the cutoff don't appear.

Why the cutoff? In the mini project, you'll predict what customers do *after* a date, using only what was known *before* it. Letting even one later order into the features would leak the future into the model.

In [ ]:
import pandas as pd


def rfm_features(orders: pd.DataFrame, cutoff: pd.Timestamp) -> pd.DataFrame:
    # your code here
    pass

In [ ]:
check_rfm_features(rfm_features)

### Exercise 12: `monthly_pivot`

Write `monthly_pivot(orders)` that turns a table of orders (columns `customer_id`, `order_date`, `amount`) into one row per customer and one column per month:

- the index is `customer_id`, sorted
- the columns are month labels like `"2025-01"`, for **every** month from the first order's month to the last order's month, including months with no orders at all, in order
- name the columns axis `"month"`
- each cell is that customer's total spending in that month, with `0.0` where there's nothing

This "wide" layout is a common input format for forecasting and churn models.

In [ ]:
import pandas as pd


def monthly_pivot(orders: pd.DataFrame) -> pd.DataFrame:
    # your code here
    pass

In [ ]:
check_monthly_pivot(monthly_pivot)

## Done?

Once everything passes, compare with `03_solutions.ipynb`. Then head to the mini project in `04_mini_project.ipynb`, where joins, `groupby` and the RFM features come together to predict which customers will stop buying.